In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/legacyagent'
for sub in ['raw_audio', 'raw_transcripts', 'processed', 'checkpoints', 'eval_results']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!nvidia-smi
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

Sun Jul  5 12:48:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import subprocess, os
repo_path = f'{PROJECT_DIR}/supreme_court_transcripts'
if not os.path.exists(repo_path):
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/walkerdb/supreme_court_transcripts.git', repo_path])
    subprocess.run(['git', 'sparse-checkout', 'set', '--no-cone', 'oyez/case_summaries.json'], cwd=repo_path)

In [ ]:
import json
d = json.load(open(f'{PROJECT_DIR}/supreme_court_transcripts/oyez/case_summaries.json'))
matches = [c for c in d if 'qualified immunity' in ((c.get('question') or '') + (c.get('description') or '')).lower()]
print(len(matches), 'matches')
for c in matches:
    print(c['term'], c['docket_number'], c['name'])

26 matches
1977 76-709 Butz v. Economou
1994 94-455 Johnson v. Jones
1995 94-1244 Behrens v. Pelletier
1996 96-292 Johnson v. Fankell
1996 96-318 Richardson v. McKnight
2000 99-1977 Saucier v. Katz
2004 03-1261 Brosseau v. Haugen
2004 03-710 Devenpeck v. Alford
2006 06-278 Morse v. Frederick
2008 07-751 Pearson, et al. v. Callahan
2008 07-1015 Ashcroft v. Iqbal
2010 10-98 Ashcroft v. Al-Kidd
2011 11-262 Reichle v. Howards
2011 10-1018 Filarsky v. Delia
2011 10-704 Messerschmidt v. Millender
2013 12-1217 Stanton v. Sims
2013 13-115 Wood v. Moss
2013 12-1117 Plumhoff v. Rickard
2013 13-483 Lane v. Franks
2014 14-212 Carroll v. Carman
2013 13-551 Tolan v. Cotton
2001 01-309 Hope v. Pelzer
2015 14-1143 Mullenix v. Luna
2016 15-1358 Ziglar v. Abbasi
2016 15-118 Hernandez v. Mesa
2017 15-1485 District of Columbia v. Wesby


In [ ]:
subprocess.run(['git', 'sparse-checkout', 'add'] +
    [f'oyez/cases/{c["term"]}.{c["docket_number"]}.json' for c in matches] +
    [f'oyez/cases/{c["term"]}.{c["docket_number"]}-t01.json' for c in matches],
    cwd=repo_path)

# sanity check — how many actually landed on disk
import os
downloaded = [c for c in matches if os.path.exists(f'{repo_path}/oyez/cases/{c["term"]}.{c["docket_number"]}-t01.json')]
print(f'{len(downloaded)}/{len(matches)} transcript files downloaded')

21/26 transcript files downloaded


In [10]:
import random
import json
import os

# ---------------------------------------------------------
# 1. Build the list of genuinely usable cases
# ---------------------------------------------------------

valid_cases = []

for c in matches:
    tpath = (
        f'{repo_path}/oyez/cases/'
        f'{c["term"]}.{c["docket_number"]}-t01.json'
    )

    if not os.path.exists(tpath):
        continue

    with open(tpath, "r") as f:
        t = json.load(f)

    if t.get("unavailable") or t.get("damaged"):
        continue

    valid_cases.append({
        "term": c["term"],
        "docket": c["docket_number"],
        "name": c["name"],
        "audio_url": t["media_file"][0]["href"],
        "duration": t["transcript"]["duration"]
    })


print("Total usable cases:", len(valid_cases))


# ---------------------------------------------------------
# 2. Deterministic shuffle
# ---------------------------------------------------------

random.seed(42)
random.shuffle(valid_cases)


# ---------------------------------------------------------
# 3. Case-level split
# ---------------------------------------------------------

# FINAL TEST SET
# Never use these cases for:
# - training
# - checkpoint selection
# - hyperparameter decisions
test_set = valid_cases[:5]


# Remaining 15 development cases
development_set = valid_cases[5:20]


# Model learns from these
train_set = development_set[:12]


# Used for checkpoint/model decisions
validation_set = development_set[12:15]


# ---------------------------------------------------------
# 4. Freeze the split permanently
# ---------------------------------------------------------

case_list = {
    "seed": 42,
    "doctrine": "qualified immunity",

    "train_set": train_set,
    "validation_set": validation_set,
    "test_set": test_set
}

case_list_path = f"{PROJECT_DIR}/case_list.json"

with open(case_list_path, "w") as f:
    json.dump(case_list, f, indent=2)


# ---------------------------------------------------------
# 5. Sanity check
# ---------------------------------------------------------

print("\nFINAL CASE-LEVEL SPLIT")
print("----------------------")
print("Train cases:     ", len(train_set))
print("Validation cases:", len(validation_set))
print("Test cases:      ", len(test_set))
print("Total cases:     ",
      len(train_set) + len(validation_set) + len(test_set))

print("\nSaved to:")
print(case_list_path)

Total usable cases: 21

FINAL CASE-LEVEL SPLIT
----------------------
Train cases:      12
Validation cases: 3
Test cases:       5
Total cases:      20

Saved to:
/content/drive/MyDrive/legacyagent/case_list.json


In [11]:
!pip install -q pydub

import os
import urllib.request
from pydub import AudioSegment

# All 20 frozen cases
all_cases = train_set + validation_set + test_set

print(f"Total cases to verify: {len(all_cases)}")

for i, case in enumerate(all_cases, start=1):

    case_id = f'{case["term"]}_{case["docket"]}'

    mp3_path = (
        f'{PROJECT_DIR}/raw_audio/'
        f'{case_id}.mp3'
    )

    wav_path = (
        f'{PROJECT_DIR}/raw_audio/'
        f'{case_id}.wav'
    )

    print(f"\n[{i}/{len(all_cases)}] {case['name']}")

    # Skip work already completed in your previous run
    if os.path.exists(wav_path):
        print("✓ WAV already exists — skipping")
        continue

    print("Downloading MP3...")
    urllib.request.urlretrieve(
        case["audio_url"],
        mp3_path
    )

    print("Converting to 16 kHz mono WAV...")

    audio = AudioSegment.from_file(mp3_path)

    audio = (
        audio
        .set_frame_rate(16000)
        .set_channels(1)
    )

    audio.export(
        wav_path,
        format="wav"
    )

    print("✓ Saved:", wav_path)

print("\n✓ Audio preparation complete")

Total cases to verify: 20

[1/20] Messerschmidt v. Millender
✓ WAV already exists — skipping

[2/20] Plumhoff v. Rickard
✓ WAV already exists — skipping

[3/20] Ziglar v. Abbasi
✓ WAV already exists — skipping

[4/20] Devenpeck v. Alford
✓ WAV already exists — skipping

[5/20] Filarsky v. Delia
✓ WAV already exists — skipping

[6/20] Hope v. Pelzer
✓ WAV already exists — skipping

[7/20] Ashcroft v. Al-Kidd
✓ WAV already exists — skipping

[8/20] Johnson v. Jones
✓ WAV already exists — skipping

[9/20] Reichle v. Howards
✓ WAV already exists — skipping

[10/20] Behrens v. Pelletier
✓ WAV already exists — skipping

[11/20] Lane v. Franks
✓ WAV already exists — skipping

[12/20] Morse v. Frederick
✓ WAV already exists — skipping

[13/20] Pearson, et al. v. Callahan
✓ WAV already exists — skipping

[14/20] Butz v. Economou
✓ WAV already exists — skipping

[15/20] Johnson v. Fankell
✓ WAV already exists — skipping

[16/20] Hernandez v. Mesa
✓ WAV already exists — skipping

[17/20] Saucier 

In [12]:
import json
import os

# Use the first TRAIN case — never inspect test data for development decisions
sample_case = train_set[0]

term = sample_case["term"]
docket = sample_case["docket"]

transcript_path = (
    f"{repo_path}/oyez/cases/"
    f"{term}.{docket}-t01.json"
)

print("CASE:", sample_case["name"])
print("TRANSCRIPT:", transcript_path)
print("EXISTS:", os.path.exists(transcript_path))

with open(transcript_path, "r") as f:
    hearing = json.load(f)

print("\nTOP-LEVEL KEYS")
print(list(hearing.keys()))

print("\nTRANSCRIPT KEYS")
print(list(hearing["transcript"].keys()))

sections = hearing["transcript"]["sections"]

print("\nNUMBER OF SECTIONS:", len(sections))

first_section = sections[0]

print("\nFIRST SECTION KEYS")
print(list(first_section.keys()))

turns = first_section.get("turns", [])

print("\nNUMBER OF TURNS IN FIRST SECTION:", len(turns))

first_turn = turns[0]

print("\nFIRST TURN KEYS")
print(list(first_turn.keys()))

print("\nSPEAKER")
print(json.dumps(first_turn.get("speaker"), indent=2))

blocks = first_turn.get("text_blocks", [])

print("\nNUMBER OF TEXT BLOCKS IN FIRST TURN:", len(blocks))

print("\nFIRST 5 TEXT BLOCKS")
for block in blocks[:5]:
    print(json.dumps(block, indent=2))
    print("-" * 60)

CASE: Messerschmidt v. Millender
TRANSCRIPT: /content/drive/MyDrive/legacyagent/supreme_court_transcripts/oyez/cases/2011.10-704-t01.json
EXISTS: True

TOP-LEVEL KEYS
['unavailable', 'public_note', 'transcript', 'media_file', 'id', 'damaged', 'title']

TRANSCRIPT KEYS
['duration', 'sections', 'title']

NUMBER OF SECTIONS: 4

FIRST SECTION KEYS
['start', 'turns', 'byte_start', 'stop', 'byte_stop']

NUMBER OF TURNS IN FIRST SECTION: 100

FIRST TURN KEYS
['byte_start', 'stop', 'byte_stop', 'text_blocks', 'start', 'speaker']

SPEAKER
{
  "identifier": "john_g_roberts_jr",
  "view_count": 106,
  "length_of_service": 3842,
  "roles": [
    {
      "institution_name": "Supreme Court of the United States",
      "type": "scotus_justice",
      "appointing_president": "George W. Bush",
      "date_end": 0,
      "date_start": 1127970000,
      "id": 2730,
      "href": "https://api.oyez.org/preson_role/scotus_justice/2730",
      "role_title": "Chief Justice of the United States"
    }
  ],
  "

In [13]:
import json
import os
import statistics

# Development data only.
# We do NOT inspect test-set distributions while designing segmentation.
development_cases = train_set + validation_set

block_durations = []
missing_start = 0
missing_stop = 0
empty_text = 0
invalid_duration = 0
total_blocks = 0

per_case_stats = []

for case in development_cases:

    transcript_path = (
        f"{repo_path}/oyez/cases/"
        f'{case["term"]}.{case["docket"]}-t01.json'
    )

    with open(transcript_path, "r") as f:
        hearing = json.load(f)

    case_durations = []

    for section in hearing["transcript"]["sections"]:

        for turn in section.get("turns", []):

            for block in turn.get("text_blocks", []):

                total_blocks += 1

                text = (block.get("text") or "").strip()
                start = block.get("start")
                stop = block.get("stop")

                if not text:
                    empty_text += 1

                if start is None:
                    missing_start += 1
                    continue

                if stop is None:
                    missing_stop += 1
                    continue

                duration = stop - start

                if duration <= 0:
                    invalid_duration += 1
                    continue

                block_durations.append(duration)
                case_durations.append(duration)

    per_case_stats.append({
        "case": case["name"],
        "blocks": len(case_durations),
        "hours": sum(case_durations) / 3600
    })


print("RAW TEXT-BLOCK AUDIT")
print("--------------------")
print("Development cases:", len(development_cases))
print("Total blocks:", total_blocks)
print("Valid timed blocks:", len(block_durations))
print("Missing start:", missing_start)
print("Missing stop:", missing_stop)
print("Empty text:", empty_text)
print("Invalid duration:", invalid_duration)


print("\nDURATION DISTRIBUTION")
print("---------------------")
print("Minimum:", round(min(block_durations), 3), "sec")
print("Median:", round(statistics.median(block_durations), 3), "sec")
print("Mean:", round(statistics.mean(block_durations), 3), "sec")
print("Maximum:", round(max(block_durations), 3), "sec")


def count_where(condition):
    return sum(1 for d in block_durations if condition(d))


print("\nDURATION BUCKETS")
print("----------------")
print("< 1 sec:    ", count_where(lambda d: d < 1))
print("1–3 sec:    ", count_where(lambda d: 1 <= d < 3))
print("3–10 sec:   ", count_where(lambda d: 3 <= d < 10))
print("10–20 sec:  ", count_where(lambda d: 10 <= d < 20))
print("20–30 sec:  ", count_where(lambda d: 20 <= d <= 30))
print("> 30 sec:   ", count_where(lambda d: d > 30))


print("\nPER-CASE COVERAGE")
print("-----------------")

for row in per_case_stats:
    print(
        f'{row["case"]}: '
        f'{row["blocks"]} blocks, '
        f'{row["hours"]:.2f} timed hours'
    )

RAW TEXT-BLOCK AUDIT
--------------------
Development cases: 15
Total blocks: 7997
Valid timed blocks: 7993
Missing start: 0
Missing stop: 0
Empty text: 0
Invalid duration: 4

DURATION DISTRIBUTION
---------------------
Minimum: 0.037 sec
Median: 4.554 sec
Mean: 6.585 sec
Maximum: 80.505 sec

DURATION BUCKETS
----------------
< 1 sec:     937
1–3 sec:     2013
3–10 sec:    3303
10–20 sec:   1387
20–30 sec:   271
> 30 sec:    82

PER-CASE COVERAGE
-----------------
Messerschmidt v. Millender: 626 blocks, 1.02 timed hours
Plumhoff v. Rickard: 698 blocks, 1.02 timed hours
Ziglar v. Abbasi: 371 blocks, 0.96 timed hours
Devenpeck v. Alford: 548 blocks, 0.92 timed hours
Filarsky v. Delia: 657 blocks, 1.01 timed hours
Hope v. Pelzer: 387 blocks, 0.95 timed hours
Ashcroft v. Al-Kidd: 464 blocks, 0.95 timed hours
Johnson v. Jones: 419 blocks, 0.88 timed hours
Reichle v. Howards: 595 blocks, 0.99 timed hours
Behrens v. Pelletier: 606 blocks, 0.96 timed hours
Lane v. Franks: 497 blocks, 0.93 time

In [14]:
import json
import statistics

TARGET_MIN_SECONDS = 8.0
TARGET_MAX_SECONDS = 25.0
HARD_MAX_SECONDS = 30.0


def get_speaker_id(speaker):
    if not speaker:
        return "unknown"

    return (
        speaker.get("identifier")
        or speaker.get("name")
        or "unknown"
    )


def simulate_segments(cases):

    segments = []

    for case in cases:

        transcript_path = (
            f"{repo_path}/oyez/cases/"
            f'{case["term"]}.{case["docket"]}-t01.json'
        )

        with open(transcript_path, "r") as f:
            hearing = json.load(f)

        for section in hearing["transcript"]["sections"]:

            for turn in section.get("turns", []):

                speaker = turn.get("speaker") or {}
                speaker_id = get_speaker_id(speaker)

                current_blocks = []

                for block in turn.get("text_blocks", []):

                    text = (block.get("text") or "").strip()
                    start = block.get("start")
                    stop = block.get("stop")

                    if not text:
                        continue

                    if start is None or stop is None:
                        continue

                    duration = stop - start

                    if duration <= 0:
                        continue

                    # -----------------------------------------
                    # Long raw block:
                    # keep separately for now so we can audit it
                    # -----------------------------------------

                    if duration > HARD_MAX_SECONDS:

                        if current_blocks:
                            segments.append({
                                "case": case["name"],
                                "speaker": speaker_id,
                                "start": current_blocks[0]["start"],
                                "stop": current_blocks[-1]["stop"],
                                "text": " ".join(
                                    b["text"] for b in current_blocks
                                )
                            })

                            current_blocks = []

                        segments.append({
                            "case": case["name"],
                            "speaker": speaker_id,
                            "start": start,
                            "stop": stop,
                            "text": text,
                            "long_raw_block": True
                        })

                        continue

                    # -----------------------------------------
                    # First block in a new segment
                    # -----------------------------------------

                    if not current_blocks:
                        current_blocks = [block]
                        continue

                    proposed_start = current_blocks[0]["start"]
                    proposed_stop = stop
                    proposed_duration = (
                        proposed_stop - proposed_start
                    )

                    # -----------------------------------------
                    # Adding this block would exceed target
                    # -----------------------------------------

                    if proposed_duration > TARGET_MAX_SECONDS:

                        segments.append({
                            "case": case["name"],
                            "speaker": speaker_id,
                            "start": current_blocks[0]["start"],
                            "stop": current_blocks[-1]["stop"],
                            "text": " ".join(
                                b["text"] for b in current_blocks
                            )
                        })

                        current_blocks = [block]

                    else:
                        current_blocks.append(block)

                # Flush final blocks from this speaker turn
                if current_blocks:

                    segments.append({
                        "case": case["name"],
                        "speaker": speaker_id,
                        "start": current_blocks[0]["start"],
                        "stop": current_blocks[-1]["stop"],
                        "text": " ".join(
                            b["text"] for b in current_blocks
                        )
                    })

    return segments


simulated_segments = simulate_segments(development_cases)

segment_durations = [
    s["stop"] - s["start"]
    for s in simulated_segments
]

print("SIMULATED SEGMENTATION")
print("----------------------")
print("Segments:", len(simulated_segments))

print("\nDURATION DISTRIBUTION")
print("---------------------")
print(
    "Minimum:",
    round(min(segment_durations), 3),
    "sec"
)
print(
    "Median:",
    round(statistics.median(segment_durations), 3),
    "sec"
)
print(
    "Mean:",
    round(statistics.mean(segment_durations), 3),
    "sec"
)
print(
    "Maximum:",
    round(max(segment_durations), 3),
    "sec"
)


def count_where(condition):
    return sum(
        1 for d in segment_durations
        if condition(d)
    )


print("\nDURATION BUCKETS")
print("----------------")
print("< 1 sec:   ", count_where(lambda d: d < 1))
print("1–3 sec:   ", count_where(lambda d: 1 <= d < 3))
print("3–8 sec:   ", count_where(lambda d: 3 <= d < 8))
print("8–15 sec:  ", count_where(lambda d: 8 <= d < 15))
print("15–25 sec: ", count_where(lambda d: 15 <= d <= 25))
print("> 25 sec:  ", count_where(lambda d: d > 25))


long_raw = [
    s for s in simulated_segments
    if s.get("long_raw_block")
]

print("\nRAW BLOCKS OVER 30 SEC")
print("----------------------")
print(len(long_raw))


print("\nFIRST 10 SIMULATED SEGMENTS")
print("---------------------------")

for segment in simulated_segments[:10]:

    duration = segment["stop"] - segment["start"]

    print(
        f'\n{duration:.2f}s | '
        f'{segment["speaker"]}'
    )

    print(segment["text"])

SIMULATED SEGMENTATION
----------------------
Segments: 4605

DURATION DISTRIBUTION
---------------------
Minimum: 0.051 sec
Median: 10.098 sec
Mean: 11.427 sec
Maximum: 80.505 sec

DURATION BUCKETS
----------------
< 1 sec:    431
1–3 sec:    674
3–8 sec:    884
8–15 sec:   959
15–25 sec:  1497
> 25 sec:   160

RAW BLOCKS OVER 30 SEC
----------------------
82

FIRST 10 SIMULATED SEGMENTS
---------------------------

9.02s | john_g_roberts_jr
We will hear argument next in Case 10-704, Messerschmidt v. Millender. Mr. Coates.

21.51s | timothy_t_coates
Mr. Chief Justice, and may it please the Court: In Malley v. Briggs and United States v. Leon, this Court set forth a very high standard for denying qualified immunity in the civil context or suppressing evidence in the criminal context under circumstances where a police officer has procured a warrant that is subsequently determined to be invalid.

31.19s | timothy_t_coates
Specifically, the Court held that the initial magistrate's determi

In [15]:
tiny_segments = []
long_segments = []

for segment in simulated_segments:
    duration = segment["stop"] - segment["start"]

    if duration < 3.0:
        tiny_segments.append(segment)

    if duration > 30.0:
        long_segments.append(segment)


print("SEGMENTATION FAILURE AUDIT")
print("--------------------------")
print("Tiny segments (<3 sec):", len(tiny_segments))
print("Long segments (>30 sec):", len(long_segments))


tiny_word_counts = [
    len(s["text"].split())
    for s in tiny_segments
]

print("\nTINY SEGMENT WORD COUNTS")
print("------------------------")
print("1 word:",
      sum(1 for n in tiny_word_counts if n == 1))
print("2–3 words:",
      sum(1 for n in tiny_word_counts if 2 <= n <= 3))
print("4–7 words:",
      sum(1 for n in tiny_word_counts if 4 <= n <= 7))
print("8+ words:",
      sum(1 for n in tiny_word_counts if n >= 8))


long_word_counts = [
    len(s["text"].split())
    for s in long_segments
]

print("\nLONG SEGMENT WORD COUNTS")
print("------------------------")
print("Minimum words:", min(long_word_counts))
print("Maximum words:", max(long_word_counts))
print(
    "Average words:",
    round(sum(long_word_counts) / len(long_word_counts), 1)
)


print("\nFIRST 15 TINY SEGMENTS")
print("----------------------")

for segment in tiny_segments[:15]:
    duration = segment["stop"] - segment["start"]

    print(
        f'\n{duration:.2f}s | '
        f'{segment["speaker"]}'
    )

    print(repr(segment["text"]))


print("\nFIRST 10 LONG SEGMENTS")
print("----------------------")

for segment in long_segments[:10]:
    duration = segment["stop"] - segment["start"]

    print(
        f'\n{duration:.2f}s | '
        f'{segment["speaker"]} | '
        f'{len(segment["text"].split())} words'
    )

    print(segment["text"])

SEGMENTATION FAILURE AUDIT
--------------------------
Tiny segments (<3 sec): 1105
Long segments (>30 sec): 82

TINY SEGMENT WORD COUNTS
------------------------
1 word: 217
2–3 words: 295
4–7 words: 368
8+ words: 225

LONG SEGMENT WORD COUNTS
------------------------
Minimum words: 2
Maximum words: 233
Average words: 81.7

FIRST 15 TINY SEGMENTS
----------------------

1.40s | timothy_t_coates
'Correct. There--'

1.70s | john_g_roberts_jr
'Have we addressed that in a prior case?'

0.58s | timothy_t_coates
"No. I'm just saying that--"

0.55s | timothy_t_coates
'--No.'

2.78s | timothy_t_coates
'Oh, I mean, this is not per se a gang crime.'

1.27s | sonia_sotomayor
'This is almost like--'

1.55s | timothy_t_coates
'--The assault, correct.'

0.50s | sonia_sotomayor
'The assault--'

0.83s | sonia_sotomayor
'Counsel--'

1.68s | john_g_roberts_jr
'Whose house -- whose house was this?'

2.70s | timothy_t_coates
"Augusta Millender's house, Ms. Millender's home."

1.22s | john_g_roberts_jr
"It

In [16]:
import json
import os

MANIFEST_DIR = f"{PROJECT_DIR}/manifests"
os.makedirs(MANIFEST_DIR, exist_ok=True)

TARGET_MAX_SECONDS = 25.0
HARD_MAX_SECONDS = 30.0


def get_speaker_info(speaker):
    speaker = speaker or {}

    speaker_id = (
        speaker.get("identifier")
        or speaker.get("name")
        or "unknown"
    )

    speaker_name = speaker.get("name") or "unknown"

    roles = speaker.get("roles") or []

    speaker_role = (
        roles[0].get("type")
        if roles
        else "unknown"
    )

    return speaker_id, speaker_name, speaker_role


def build_segments(cases, split_name):

    rows = []
    flagged_long = 0
    skipped_invalid = 0

    for case in cases:

        transcript_path = (
            f"{repo_path}/oyez/cases/"
            f'{case["term"]}.{case["docket"]}-t01.json'
        )

        wav_path = (
            f"{PROJECT_DIR}/raw_audio/"
            f'{case["term"]}_{case["docket"]}.wav'
        )

        with open(transcript_path, "r") as f:
            hearing = json.load(f)

        segment_index = 0

        for section_index, section in enumerate(
            hearing["transcript"]["sections"]
        ):

            for turn_index, turn in enumerate(
                section.get("turns", [])
            ):

                speaker_id, speaker_name, speaker_role = (
                    get_speaker_info(turn.get("speaker"))
                )

                current_blocks = []

                def flush_current():
                    nonlocal segment_index

                    if not current_blocks:
                        return

                    start = current_blocks[0]["start"]
                    end = current_blocks[-1]["stop"]

                    rows.append({
                        "segment_id": (
                            f'{case["term"]}_'
                            f'{case["docket"]}_'
                            f'{segment_index:06d}'
                        ),
                        "case_id": (
                            f'{case["term"]}_{case["docket"]}'
                        ),
                        "case_name": case["name"],
                        "audio_path": wav_path,
                        "start": float(start),
                        "end": float(end),
                        "duration": float(end - start),
                        "text": " ".join(
                            b["text"] for b in current_blocks
                        ),
                        "speaker_id": speaker_id,
                        "speaker_name": speaker_name,
                        "speaker_role": speaker_role,
                        "section_index": section_index,
                        "turn_index": turn_index,
                        "split": split_name,
                        "needs_alignment": False
                    })

                    segment_index += 1
                    current_blocks.clear()

                for block in turn.get("text_blocks", []):

                    text = (block.get("text") or "").strip()
                    start = block.get("start")
                    stop = block.get("stop")

                    if (
                        not text
                        or start is None
                        or stop is None
                        or stop <= start
                    ):
                        skipped_invalid += 1
                        continue

                    duration = stop - start

                    # Long raw block: do not invent timestamps
                    if duration > HARD_MAX_SECONDS:

                        flush_current()

                        rows.append({
                            "segment_id": (
                                f'{case["term"]}_'
                                f'{case["docket"]}_'
                                f'{segment_index:06d}'
                            ),
                            "case_id": (
                                f'{case["term"]}_{case["docket"]}'
                            ),
                            "case_name": case["name"],
                            "audio_path": wav_path,
                            "start": float(start),
                            "end": float(stop),
                            "duration": float(duration),
                            "text": text,
                            "speaker_id": speaker_id,
                            "speaker_name": speaker_name,
                            "speaker_role": speaker_role,
                            "section_index": section_index,
                            "turn_index": turn_index,
                            "split": split_name,
                            "needs_alignment": True
                        })

                        segment_index += 1
                        flagged_long += 1
                        continue

                    if not current_blocks:
                        current_blocks.append(block)
                        continue

                    proposed_duration = (
                        stop - current_blocks[0]["start"]
                    )

                    if proposed_duration > TARGET_MAX_SECONDS:
                        flush_current()
                        current_blocks.append(block)
                    else:
                        current_blocks.append(block)

                flush_current()

    output_path = (
        f"{MANIFEST_DIR}/{split_name}_manifest.jsonl"
    )

    with open(output_path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    print(f"\n{split_name.upper()}")
    print("Segments:", len(rows))
    print("Long blocks flagged:", flagged_long)
    print("Invalid blocks skipped:", skipped_invalid)
    print("Saved:", output_path)

    return rows


train_manifest = build_segments(train_set, "train")

validation_manifest = build_segments(
    validation_set,
    "validation"
)

test_manifest = build_segments(
    test_set,
    "test"
)


TRAIN
Segments: 3702
Long blocks flagged: 64
Invalid blocks skipped: 3
Saved: /content/drive/MyDrive/legacyagent/manifests/train_manifest.jsonl

VALIDATION
Segments: 903
Long blocks flagged: 18
Invalid blocks skipped: 1
Saved: /content/drive/MyDrive/legacyagent/manifests/validation_manifest.jsonl

TEST
Segments: 1537
Long blocks flagged: 22
Invalid blocks skipped: 4
Saved: /content/drive/MyDrive/legacyagent/manifests/test_manifest.jsonl


In [17]:
import os
import statistics

all_manifests = {
    "train": train_manifest,
    "validation": validation_manifest,
    "test": test_manifest
}

print("MANIFEST INTEGRITY CHECK")
print("========================")

# ---------------------------------------------------------
# 1. Validate every row
# ---------------------------------------------------------

for split_name, rows in all_manifests.items():

    errors = []

    for row in rows:

        if not row["text"].strip():
            errors.append(
                (row["segment_id"], "empty text")
            )

        if row["end"] <= row["start"]:
            errors.append(
                (row["segment_id"], "invalid timestamps")
            )

        if row["duration"] <= 0:
            errors.append(
                (row["segment_id"], "invalid duration")
            )

        if not os.path.exists(row["audio_path"]):
            errors.append(
                (row["segment_id"], "missing audio")
            )

    print(f"\n{split_name.upper()}")
    print("Rows:", len(rows))
    print("Integrity errors:", len(errors))

    if errors:
        print("First 10 errors:")
        for error in errors[:10]:
            print(error)


# ---------------------------------------------------------
# 2. Check case leakage
# ---------------------------------------------------------

train_cases = {
    row["case_id"]
    for row in train_manifest
}

validation_cases = {
    row["case_id"]
    for row in validation_manifest
}

test_cases = {
    row["case_id"]
    for row in test_manifest
}

print("\nCASE LEAKAGE CHECK")
print("------------------")

print(
    "Train ∩ Validation:",
    train_cases & validation_cases
)

print(
    "Train ∩ Test:",
    train_cases & test_cases
)

print(
    "Validation ∩ Test:",
    validation_cases & test_cases
)

assert train_cases.isdisjoint(validation_cases)
assert train_cases.isdisjoint(test_cases)
assert validation_cases.isdisjoint(test_cases)

print("✓ No case leakage")


# ---------------------------------------------------------
# 3. Check duplicate segment IDs
# ---------------------------------------------------------

all_ids = [
    row["segment_id"]
    for rows in all_manifests.values()
    for row in rows
]

print("\nSEGMENT ID CHECK")
print("----------------")

print("Total IDs:", len(all_ids))
print("Unique IDs:", len(set(all_ids)))

assert len(all_ids) == len(set(all_ids))

print("✓ All segment IDs are unique")


# ---------------------------------------------------------
# 4. Separate training-ready vs flagged
# ---------------------------------------------------------

print("\nTRAINING READINESS")
print("------------------")

for split_name, rows in all_manifests.items():

    ready = [
        row for row in rows
        if not row["needs_alignment"]
    ]

    flagged = [
        row for row in rows
        if row["needs_alignment"]
    ]

    ready_hours = sum(
        row["duration"]
        for row in ready
    ) / 3600

    print(f"\n{split_name.upper()}")
    print("Training-ready:", len(ready))
    print("Flagged long:", len(flagged))
    print(
        "Ready audio hours:",
        round(ready_hours, 2)
    )

MANIFEST INTEGRITY CHECK

TRAIN
Rows: 3702
Integrity errors: 0

VALIDATION
Rows: 903
Integrity errors: 0

TEST
Rows: 1537
Integrity errors: 0

CASE LEAKAGE CHECK
------------------
Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()
✓ No case leakage

SEGMENT ID CHECK
----------------
Total IDs: 6142
Unique IDs: 6142
✓ All segment IDs are unique

TRAINING READINESS
------------------

TRAIN
Training-ready: 3638
Flagged long: 64
Ready audio hours: 10.89

VALIDATION
Training-ready: 885
Flagged long: 18
Ready audio hours: 2.82

TEST
Training-ready: 1515
Flagged long: 22
Ready audio hours: 4.76


In [18]:
import json
import statistics
from datetime import datetime, timezone

def build_split_report(rows):

    ready = [
        row for row in rows
        if not row["needs_alignment"]
    ]

    flagged = [
        row for row in rows
        if row["needs_alignment"]
    ]

    durations = [
        row["duration"]
        for row in ready
    ]

    return {
        "cases": len({
            row["case_id"]
            for row in rows
        }),

        "total_segments": len(rows),

        "training_ready_segments": len(ready),

        "flagged_long_segments": len(flagged),

        "ready_audio_hours": round(
            sum(durations) / 3600,
            2
        ),

        "min_duration_seconds": round(
            min(durations),
            3
        ),

        "median_duration_seconds": round(
            statistics.median(durations),
            3
        ),

        "mean_duration_seconds": round(
            statistics.mean(durations),
            3
        ),

        "max_duration_seconds": round(
            max(durations),
            3
        ),

        "total_words": sum(
            len(row["text"].split())
            for row in ready
        )
    }


dataset_report = {

    "project": "LegacyAgent V1",

    "dataset_source": (
        "Oyez data accessed through "
        "walkerdb/supreme_court_transcripts"
    ),

    "doctrine": "qualified immunity",

    "split_seed": 42,

    "split_policy": (
        "case-disjoint: 12 train, "
        "3 validation, 5 test"
    ),

    "segmentation_policy": {
        "merge_scope": (
            "consecutive text blocks "
            "within the same speaker turn"
        ),
        "target_max_seconds": 25.0,
        "hard_max_seconds": 30.0,
        "tiny_turn_policy": (
            "preserve genuine short speaker turns"
        ),
        "long_block_policy": (
            "flag as needs_alignment and "
            "exclude from V1 training/evaluation"
        )
    },

    "train": build_split_report(
        train_manifest
    ),

    "validation": build_split_report(
        validation_manifest
    ),

    "test": build_split_report(
        test_manifest
    ),

    "created_at": datetime.now(
        timezone.utc
    ).isoformat()
}


report_path = (
    f"{PROJECT_DIR}/dataset_report.json"
)

with open(report_path, "w") as f:
    json.dump(
        dataset_report,
        f,
        indent=2
    )


print(
    json.dumps(
        dataset_report,
        indent=2
    )
)

print("\n✓ Saved to:")
print(report_path)

{
  "project": "LegacyAgent V1",
  "dataset_source": "Oyez data accessed through walkerdb/supreme_court_transcripts",
  "doctrine": "qualified immunity",
  "split_seed": 42,
  "split_policy": "case-disjoint: 12 train, 3 validation, 5 test",
  "segmentation_policy": {
    "merge_scope": "consecutive text blocks within the same speaker turn",
    "target_max_seconds": 25.0,
    "hard_max_seconds": 30.0,
    "tiny_turn_policy": "preserve genuine short speaker turns",
    "long_block_policy": "flag as needs_alignment and exclude from V1 training/evaluation"
  },
  "train": {
    "cases": 12,
    "total_segments": 3702,
    "training_ready_segments": 3638,
    "flagged_long_segments": 64,
    "ready_audio_hours": 10.89,
    "min_duration_seconds": 0.051,
    "median_duration_seconds": 9.538,
    "mean_duration_seconds": 10.781,
    "max_duration_seconds": 29.997,
    "total_words": 111685
  },
  "validation": {
    "cases": 3,
    "total_segments": 903,
    "training_ready_segments": 885,
 

In [19]:
import sys
import torch
import subprocess

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

print("\nNVIDIA STATUS")
subprocess.run(["nvidia-smi"])

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

NVIDIA STATUS


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [20]:
!pip install -q -U \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    librosa \
    soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00


In [2]:
import torch
import transformers
import peft
import bitsandbytes as bnb

from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

print("LIBRARY CHECK")
print("-------------")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.cuda.is_available())

print("\nLOADING CONFIG ONLY")
print("-------------------")

config = AutoConfig.from_pretrained(MODEL_ID)

print("✓ Config loaded")
print("Model ID:", MODEL_ID)
print("Model type:", config.model_type)
print("Architecture:", config.architectures)
print("Config class:", type(config).__name__)

LIBRARY CHECK
-------------
PyTorch: 2.11.0+cu128
Transformers: 5.13.0
PEFT: 0.19.1
bitsandbytes: 0.49.2
CUDA: True

LOADING CONFIG ONLY
-------------------


config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

✓ Config loaded
Model ID: Qwen/Qwen3-ASR-1.7B-hf
Model type: qwen3_asr
Architecture: ['Qwen3ASRForConditionalGeneration']
Config class: Qwen3ASRConfig


In [16]:
import gc
import torch

from transformers import AutoModelForSpeechSeq2Seq

print("RELOADING CORRECT TRAINING MODEL")
print("================================")

# Remove the old PEFT-wrapped backbone model
del model
gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated after cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nLoading Qwen3-ASR conditional-generation model in 4-bit...")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
)

print("\n✓ Correct model loaded")
print("Model class:", type(model).__name__)

assert (
    type(model).__name__
    == "Qwen3ASRForConditionalGeneration"
)

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

RELOADING CORRECT TRAINING MODEL
GPU allocated after cleanup: 1.06 GB

Loading Qwen3-ASR conditional-generation model in 4-bit...


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]


✓ Correct model loaded
Model class: Qwen3ASRForConditionalGeneration
GPU allocated: 2.47 GB


In [17]:
import torch
from collections import Counter

print("MODEL STRUCTURE INSPECTION")
print("==========================")

print("Model class:", type(model).__name__)

print("\nTOP-LEVEL CHILDREN")
print("------------------")

for name, module in model.named_children():
    print(f"{name}: {type(module).__name__}")


# Find linear-like modules that LoRA could potentially target
linear_module_names = []

for name, module in model.named_modules():

    class_name = type(module).__name__

    if (
        isinstance(module, torch.nn.Linear)
        or "Linear4bit" in class_name
    ):
        linear_module_names.append(name)


print("\nTOTAL LINEAR-LIKE MODULES")
print("-------------------------")
print(len(linear_module_names))


print("\nFIRST 80 LINEAR-LIKE MODULES")
print("----------------------------")

for name in linear_module_names[:80]:
    print(name)


# Count repeated final module names
leaf_names = [
    name.split(".")[-1]
    for name in linear_module_names
]

counts = Counter(leaf_names)

print("\nREPEATED LINEAR MODULE NAMES")
print("----------------------------")

for name, count in counts.most_common():
    print(f"{name}: {count}")

MODEL STRUCTURE INSPECTION
Model class: Qwen3ASRForConditionalGeneration

TOP-LEVEL CHILDREN
------------------
model: Qwen3ASRModel
lm_head: Linear

TOTAL LINEAR-LIKE MODULES
-------------------------
344

FIRST 80 LINEAR-LIKE MODULES
----------------------------
model.audio_tower.layers.0.self_attn.k_proj
model.audio_tower.layers.0.self_attn.v_proj
model.audio_tower.layers.0.self_attn.q_proj
model.audio_tower.layers.0.self_attn.out_proj
model.audio_tower.layers.0.fc1
model.audio_tower.layers.0.fc2
model.audio_tower.layers.1.self_attn.k_proj
model.audio_tower.layers.1.self_attn.v_proj
model.audio_tower.layers.1.self_attn.q_proj
model.audio_tower.layers.1.self_attn.out_proj
model.audio_tower.layers.1.fc1
model.audio_tower.layers.1.fc2
model.audio_tower.layers.2.self_attn.k_proj
model.audio_tower.layers.2.self_attn.v_proj
model.audio_tower.layers.2.self_attn.q_proj
model.audio_tower.layers.2.self_attn.out_proj
model.audio_tower.layers.2.fc1
model.audio_tower.layers.2.fc2
model.audio_tow

In [18]:
from collections import Counter

print("LORA TARGET MAP")
print("===============")

components = {
    "audio_tower": [],
    "language_model": [],
    "multi_modal_projector": [],
    "other": []
}

for name in linear_module_names:
    if name.startswith("audio_tower."):
        components["audio_tower"].append(name)

    elif name.startswith("language_model."):
        components["language_model"].append(name)

    elif name.startswith("multi_modal_projector."):
        components["multi_modal_projector"].append(name)

    else:
        components["other"].append(name)


for component, names in components.items():

    leaf_counts = Counter(
        name.split(".")[-1]
        for name in names
    )

    print(f"\n{component.upper()}")
    print("-" * len(component))
    print("Total linear-like modules:", len(names))

    for leaf, count in leaf_counts.most_common():
        print(f"{leaf}: {count}")

LORA TARGET MAP

AUDIO_TOWER
-----------
Total linear-like modules: 0

LANGUAGE_MODEL
--------------
Total linear-like modules: 0

MULTI_MODAL_PROJECTOR
---------------------
Total linear-like modules: 0

OTHER
-----
Total linear-like modules: 344
k_proj: 52
v_proj: 52
q_proj: 52
o_proj: 28
gate_proj: 28
up_proj: 28
down_proj: 28
out_proj: 24
fc1: 24
fc2: 24
conv_out: 1
linear_1: 1
linear_2: 1
lm_head: 1


In [19]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("PREPARING 4-BIT MODEL FOR LORA")
print("==============================")

# Prepare quantized base model for gradient-based adapter training
model = prepare_model_for_kbit_training(model)

# Exact full suffixes prevent accidental attachment to audio_tower q_proj/v_proj
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "language_model.layers.0.self_attn.q_proj",
        "language_model.layers.0.self_attn.v_proj",
    ],
)

print("Attaching tiny smoke-test adapters...")

model = get_peft_model(
    model,
    lora_config
)

print("\n✓ LoRA attached")

print("\nTRAINABLE PARAMETER REPORT")
print("--------------------------")
model.print_trainable_parameters()

print("\nLORA MODULES CREATED")
print("--------------------")

lora_modules = [
    name
    for name, module in model.named_modules()
    if "lora_" in name
]

for name in lora_modules:
    print(name)

print("\nTotal LoRA submodules:", len(lora_modules))

PREPARING 4-BIT MODEL FOR LORA
Attaching tiny smoke-test adapters...

✓ LoRA attached

TRAINABLE PARAMETER REPORT
--------------------------
trainable params: 57,344 || all params: 2,038,109,824 || trainable%: 0.0028

LORA MODULES CREATED
--------------------
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_dropout
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_dropout.default
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_embedding_A
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_embedding_B
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_magnitude_vector
base_model.model.model.languag

In [20]:
import inspect

print("PROCESSOR + MODEL API INSPECTION")
print("================================")

print("\nPROCESSOR CLASS")
print("----------------")
print(type(processor).__name__)

print("\nPROCESSOR CALL SIGNATURE")
print("------------------------")
print(inspect.signature(processor.__call__))

print("\nMODEL FORWARD SIGNATURE")
print("-----------------------")
print(inspect.signature(model.forward))

print("\nPROCESSOR ATTRIBUTES")
print("--------------------")

for attr in [
    "tokenizer",
    "feature_extractor",
    "audio_processor",
    "apply_chat_template"
]:
    print(
        f"{attr}:",
        hasattr(processor, attr)
    )

PROCESSOR + MODEL API INSPECTION

PROCESSOR CLASS
----------------
Qwen3ASRProcessor

PROCESSOR CALL SIGNATURE
------------------------
(text: str | list[str], audio: Union[numpy.ndarray, ForwardRef('torch.Tensor'), collections.abc.Sequence[numpy.ndarray], collections.abc.Sequence['torch.Tensor']], output_labels: bool | None = False, **kwargs: Unpack[transformers.models.qwen3_asr.processing_qwen3_asr.Qwen3ASRProcessorKwargs]) -> transformers.feature_extraction_utils.BatchFeature

MODEL FORWARD SIGNATURE
-----------------------
(*args: 'Any', **kwargs: 'Any')

PROCESSOR ATTRIBUTES
--------------------
tokenizer: True
feature_extractor: True
audio_processor: False
apply_chat_template: True


In [21]:
import json

TRAIN_MANIFEST_PATH = (
    f"{PROJECT_DIR}/manifests/train_manifest.jsonl"
)

with open(TRAIN_MANIFEST_PATH, "r") as f:
    train_manifest = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Train manifest reloaded:", len(train_manifest))

ready_count = sum(
    not row["needs_alignment"]
    for row in train_manifest
)

print("Training-ready:", ready_count)

Train manifest reloaded: 3702
Training-ready: 3638


In [22]:
import json
import librosa
import torch

print("REAL TRAINING SAMPLE PREPARATION")
print("================================")

# Pick the first training-ready sample
sample = next(
    row for row in train_manifest
    if not row["needs_alignment"]
)

print("\nSAMPLE")
print("------")
print("Segment ID:", sample["segment_id"])
print("Case:", sample["case_name"])
print("Duration:", round(sample["duration"], 3), "sec")
print("Transcript:", sample["text"])


# Load only this timestamp range from the full hearing WAV
audio, sample_rate = librosa.load(
    sample["audio_path"],
    sr=16000,
    offset=sample["start"],
    duration=sample["duration"]
)

print("\nAUDIO")
print("-----")
print("Sample rate:", sample_rate)
print("Audio samples:", len(audio))
print(
    "Loaded duration:",
    round(len(audio) / sample_rate, 3),
    "sec"
)


# Native supervised-ASR preprocessing
inputs = processor(
    text=sample["text"],
    audio=audio,
    output_labels=True,
    return_tensors="pt"
)

print("\nPROCESSOR OUTPUT")
print("----------------")

for key, value in inputs.items():
    if isinstance(value, torch.Tensor):
        print(
            f"{key}: "
            f"shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )
    else:
        print(
            f"{key}: "
            f"{type(value).__name__}"
        )


print("\nLABEL CHECK")
print("-----------")

if "labels" in inputs:
    print("✓ Labels created")
    print("Label shape:", tuple(inputs["labels"].shape))
else:
    print("✗ No labels created")

REAL TRAINING SAMPLE PREPARATION

SAMPLE
------
Segment ID: 2011_10-704_000000
Case: Messerschmidt v. Millender
Duration: 9.022 sec
Transcript: We will hear argument next in Case 10-704, Messerschmidt v. Millender. Mr. Coates.

AUDIO
-----
Sample rate: 16000
Audio samples: 144352
Loaded duration: 9.022 sec

PROCESSOR OUTPUT
----------------
input_ids: shape=(1, 29), dtype=torch.int64
attention_mask: shape=(1, 29), dtype=torch.int64
input_features: shape=(1, 128, 1000), dtype=torch.float32
input_features_mask: shape=(1, 1000), dtype=torch.int32
labels: shape=(1, 29), dtype=torch.int64

LABEL CHECK
-----------
✓ Labels created
Label shape: (1, 29)


In [23]:
print("CORRECT AUDIO-AWARE SAMPLE")
print("==========================")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "audio",
                "audio": audio
            }
        ]
    }
]

prompt = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False
)

print("\nGENERATED PROMPT")
print("----------------")
print(repr(prompt))

correct_inputs = processor(
    text=prompt + sample["text"],
    audio=audio,
    output_labels=True,
    return_tensors="pt"
)

print("\nPROCESSOR OUTPUT")
print("----------------")

for key, value in correct_inputs.items():
    if isinstance(value, torch.Tensor):
        print(
            f"{key}: "
            f"shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )

audio_token_count = (
    correct_inputs["input_ids"]
    == processor.audio_token_id
).sum().item()

print("\nAUDIO TOKEN CHECK")
print("-----------------")
print("Audio tokens:", audio_token_count)

print("\nLABEL CHECK")
print("-----------")
print("Labels created:", "labels" in correct_inputs)
print(
    "Label shape:",
    tuple(correct_inputs["labels"].shape)
)

CORRECT AUDIO-AWARE SAMPLE

GENERATED PROMPT
----------------
'<|im_start|>system\n<|im_end|>\n<|im_start|>user\n<|audio_start|><|audio_pad|><|audio_end|><|im_end|>\n<|im_start|>assistant\n'

PROCESSOR OUTPUT
----------------
input_ids: shape=(1, 162), dtype=torch.int64
attention_mask: shape=(1, 162), dtype=torch.int64
input_features: shape=(1, 128, 1000), dtype=torch.float32
input_features_mask: shape=(1, 1000), dtype=torch.int32
labels: shape=(1, 162), dtype=torch.int64

AUDIO TOKEN CHECK
-----------------
Audio tokens: 118

LABEL CHECK
-----------
Labels created: True
Label shape: (1, 162)


In [24]:
import torch

print("FINAL QLORA SMOKE TEST — RETRY")
print("==============================")

device = next(model.parameters()).device

batch = {
    key: value.to(device)
    for key, value in correct_inputs.items()
    if isinstance(value, torch.Tensor)
}

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)

model.train()
optimizer.zero_grad()

print("\n1. Forward pass...")

outputs = model(**batch)
loss = outputs.loss

print("✓ Forward pass succeeded")
print("Loss:", float(loss.detach().cpu()))

assert torch.isfinite(loss), "Loss is not finite"

print("\n2. Backward pass...")

loss.backward()

print("✓ Backward pass succeeded")

trainable_with_grad = []
trainable_without_grad = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    if param.grad is not None:
        trainable_with_grad.append(name)
    else:
        trainable_without_grad.append(name)

print("\n3. Gradient check...")
print(
    "Trainable tensors with gradients:",
    len(trainable_with_grad)
)
print(
    "Trainable tensors without gradients:",
    len(trainable_without_grad)
)

for name in trainable_with_grad[:10]:
    print("✓", name)

assert len(trainable_with_grad) > 0, (
    "No LoRA parameters received gradients"
)

print("\n4. Optimizer step...")

optimizer.step()

print("✓ Optimizer step succeeded")

print("\nGPU MEMORY")
print("----------")
print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print(
    "Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

print("\n==============================")
print("QLORA COMPATIBILITY: PASS ✓")
print("==============================")

FINAL QLORA SMOKE TEST — RETRY

1. Forward pass...
✓ Forward pass succeeded
Loss: 19.643640518188477

2. Backward pass...
✓ Backward pass succeeded

3. Gradient check...
Trainable tensors with gradients: 4
Trainable tensors without gradients: 0
✓ base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight
✓ base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight
✓ base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight
✓ base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight

4. Optimizer step...
✓ Optimizer step succeeded

GPU MEMORY
----------
Allocated: 3.28 GB
Reserved: 3.75 GB

QLORA COMPATIBILITY: PASS ✓


In [25]:
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSpeechSeq2Seq,
)

print("AUTOCLASS MAPPING CHECK")
print("=======================")

for cls in [
    AutoModelForCausalLM,
    AutoModelForSpeechSeq2Seq,
]:
    try:
        mapped_class = cls._model_mapping[type(config)]
        print(
            cls.__name__,
            "→",
            mapped_class.__name__
        )
    except Exception as e:
        print(
            cls.__name__,
            "→ NOT MAPPED:",
            type(e).__name__
        )

AUTOCLASS MAPPING CHECK
AutoModelForCausalLM → NOT MAPPED: KeyError
AutoModelForSpeechSeq2Seq → Qwen3ASRForConditionalGeneration


In [26]:
import json
from datetime import datetime, timezone

compatibility_result = {
    "project": "LegacyAgent V1",
    "model": "Qwen/Qwen3-ASR-1.7B-hf",
    "decision": "GO",
    "training_method": "4-bit QLoRA",
    "hardware_tested": "Tesla T4 14.56 GB",
    "quantization": "NF4 4-bit",
    "smoke_test": {
        "real_project_sample": True,
        "forward_pass": True,
        "loss": 19.643640518188477,
        "backward_pass": True,
        "lora_gradients": True,
        "optimizer_step": True,
        "gpu_allocated_gb": 3.28
    },
    "note": (
        "Compatibility only. The layer-0 q_proj/v_proj LoRA setup "
        "was a minimal smoke test, not the final training configuration."
    ),
    "tested_at": datetime.now(timezone.utc).isoformat()
}

output_path = f"{PROJECT_DIR}/qlora_compatibility.json"

with open(output_path, "w") as f:
    json.dump(compatibility_result, f, indent=2)

print(json.dumps(compatibility_result, indent=2))
print("\n✓ Saved:", output_path)

{
  "project": "LegacyAgent V1",
  "model": "Qwen/Qwen3-ASR-1.7B-hf",
  "decision": "GO",
  "training_method": "4-bit QLoRA",
  "hardware_tested": "Tesla T4 14.56 GB",
  "quantization": "NF4 4-bit",
  "smoke_test": {
    "real_project_sample": true,
    "forward_pass": true,
    "loss": 19.643640518188477,
    "backward_pass": true,
    "lora_gradients": true,
    "optimizer_step": true,
    "gpu_allocated_gb": 3.28
  },
  "note": "Compatibility only. The layer-0 q_proj/v_proj LoRA setup was a minimal smoke test, not the final training configuration.",
  "tested_at": "2026-07-05T14:38:58.078536+00:00"
}

✓ Saved: /content/drive/MyDrive/legacyagent/qlora_compatibility.json
